# Does averaging more trials improve classification?

**CPU → Run all**, using **Zyion** for both Drive mounting and authentication. This is a separate follow-up and does not change the running pooled CNN experiment. Run it after that experiment finishes if you want to avoid two jobs reading Drive together.

**One fixed setup:** positive/negative ZuCo **SR**, saved **agent-labelled paper subset**, **cleaned EEG**, all 105 channel means over **224–304 ms**, logistic regression, **unseen readers**. We chose this setup because it gave the most promising earlier 40-trial result. This is exploratory follow-up on a previously examined dataset, not independent confirmation.

Test **1, 10, 20, …, 100** trials per same-class average. Use the **same eligible readers and same three folds** at every size. Each included reader needs at least 100 saved trials in each class; report exclusions based only on this count. Train/test readers never overlap. Words and sentences may overlap.

Within each partition, reader and class, shuffle and form disjoint groups of **exactly** the requested size. Discard leftover trials. Repeat grouping five times, keeping the folds and model settings fixed. The single-trial point is computed once. No search for the best seed, model or time window.

**Storage and time:** no extraction, cleaning, relabeling or original-source downloads. The first run must load the saved compressed reader EEG to obtain the 105 features and can take several minutes. The feature cache and all results share a new **20 MB Drive cap**. Reruns reuse the small cache; no repeated EEG reads after it is complete. A reader's temporary EEG copy stays only on Colab disk. Existing results remain unchanged.

**Important:** test class labels are used to construct the averages. Scores describe group-level separability, not individual-word prediction. The range across regroupings is **not a confidence interval or significance test**. Larger averages produce fewer test examples. This subset is the intersection with the previously SST-selected, matched and regression-cleaned paper archive, not the full annotated text pool. It includes archived rereading fixations, not first fixations only. Existing full-recording cleaning remains a methodological limitation for held-out-reader interpretation.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Setup and storage check

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_averaging_sweep'
REPO=Path('/content/word-sentiment-eeg-averaging-sweep')
CODE_REVISION='2b16c2a167343c1f00fcb0eb2d3f06130f07321b'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,20_000_000-used):
    raise RuntimeError('Reserve up to 20 MB for the feature cache and results. No EEG loading has started.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 20 MB.')
print('New output folder:',OUT)


## 2. Run the single experiment

Progress shows saved reader out of the total, then averaging size and grouping repeat. Each repeat uses three fixed held-out-reader folds. Logistic regression uses training-only imputation/scaling, C=1 and balanced class weights.

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_averaging_sweep.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='ZuCo averaging sweep: one fixed logistic setup')


## 3. Balanced accuracy versus averaging size

Shading shows the minimum and maximum across five regroupings, using the same events and readers. These are correlated descriptive repeats, not five independent experiments. The second plot shows how many test examples remain.

In [ ]:
completion=json.loads((OUT/'complete.json').read_text())
print('Run:',completion['status'])
display(Image(filename=str(OUT/'averaging_curve.png')))
scores=pd.read_csv(OUT/'scores.csv')
view=scores.copy()
for col in ['balanced_accuracy','grouping_sd','grouping_min','grouping_max','dummy_balanced_accuracy']:
    view[col]=(view[col]*100).round(2)
display(view.rename(columns={'size':'exact trials per average','grouping_sd':'SD across regroupings (percentage points)'}))


## 4. Input coverage

Counts are EEG instances per reader in the saved agent subset. At every point, the same included readers contribute both classes. Within each point, original trials never appear in two test averages or on both sides of a split.

In [ ]:
cohort=json.loads((OUT/'cohort.json').read_text())
counts=pd.DataFrame(cohort['all_reader_counts']).T.rename_axis('reader')
counts['included_every_size']=counts.index.isin(cohort['included_readers'])
display(counts)
print('Original subset events:',cohort['events_before'],'| Fixed cohort events:',cohort['events_used'])
print('Excluded readers:',cohort['excluded_readers'] or 'None')
print('Saved curve, tables, predictions, membership, split IDs and feature cache:',OUT)
print('A higher score at a larger size is an exploratory result; inspect the remaining test-example count and grouping range too.')
